# Bloque 3 — NLP y Modelo Pre-entrenado
## Corpus real: PQRS ciudadanas — datos.gov.co + Hugging Face
### Parcial Final — Machine Learning con PySpark y Docker

**Autor:** Sergio Prieto | **Fecha:** Mayo 2026

---
## Objetivo
Aplicar técnicas de NLP usando un **corpus textual colombiano real** descargado del portal de datos abiertos del Estado colombiano (datos.gov.co) y un modelo pre-entrenado de Hugging Face:

- **Parte A:** Carga del corpus real PQRS, tokenización, stop words, TTR, hapax
- **Parte B:** Vectorización TF-IDF y análisis estadístico por categoría
- **Parte C:** Clasificación supervisada sobre vectores TF-IDF
- **Parte D:** Modelo `pysentimiento/robertuito-sentiment-analysis` y comparación

## Corpus: PQRS — Aeropuerto El Dorado (datos.gov.co)
Se descargó el dataset **"PQRS"** (id: `e88e-ctba`) del portal `datos.gov.co`, que contiene **616 peticiones, quejas, reclamos y sugerencias** reales presentadas por ciudadanos ante la autoridad aeroportuaria. Cada registro incluye:
- **asunto:** texto libre describiendo la solicitud (nuestro corpus textual)
- **categoria:** clasificación administrativa (16 categorías, nuestra variable objetivo)

**Ventaja clave:** Corpus textual REAL colombiano, con lenguaje ciudadano auténtico, categorías naturales y suficiente variabilidad léxica para una clasificación no trivial.


In [1]:
import os, sys
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Fix PySpark Windows
HADOOP_HOME = os.path.join(os.environ.get("TEMP"), "hadoop_tmp")
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.makedirs(os.path.join(HADOOP_HOME, "bin"), exist_ok=True)
winutils_path = os.path.join(HADOOP_HOME, "bin", "winutils.exe")
if not os.path.exists(winutils_path):
    with open(winutils_path, "wb") as f: f.write(b"")

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, count, lit, when,
    size as spark_size, avg as spark_avg, min as spark_min, max as spark_max,
)
from pyspark.sql.types import IntegerType, StringType
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer,
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

PQRS_PATH = "../data/pqrs_colombia.csv"
OUTPUT_DIR = "../salidas"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Parte A — Preparación del corpus (Tareas 21-24)

### Tarea 21: Carga del corpus textual colombiano real
Se carga el dataset PQRS descargado de datos.gov.co. El campo `asunto` contiene el texto libre de la petición/queja/sugerencia ciudadana. La `categoria` es la clasificación administrativa.

Se unifican categorías con menos de 10 documentos en una sola categoría "Otros" para evitar clases extremadamente minoritarias.


In [2]:
spark = SparkSession.builder.appName("PQRS_Bloque3_NLP").master("local[*]") \
    .config("spark.sql.shuffle.partitions","4").config("spark.ui.enabled","false") \
    .config("spark.driver.memory","2g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Cargar corpus PQRS real
print("Cargando corpus PQRS (datos.gov.co)...")
pdf_raw = pd.read_csv(PQRS_PATH, dtype=str, keep_default_na=False)
print(f"  Registros: {len(pdf_raw):,}")
print(f"  Columnas: {list(pdf_raw.columns)}")

# Distribucion original de categorias
print("\n  Distribucion original de categorias:")
cat_counts = pdf_raw["categoria"].value_counts()
for cat, cnt in cat_counts.items():
    print(f"    {cat}: {cnt}")

# Unificar categorias pequenas (< 10 docs) en "Otros"
cats_to_keep = cat_counts[cat_counts >= 10].index.tolist()
pdf_raw["categoria"] = pdf_raw["categoria"].apply(
    lambda x: x if x in cats_to_keep else "Otros"
)
print(f"\n  Despues de unificar: {pdf_raw['categoria'].nunique()} categorias")
print(pdf_raw["categoria"].value_counts().to_string())


Cargando corpus PQRS (datos.gov.co)...
  Registros: 616
  Columnas: ['pqrsds', 'asunto', 'categoria', 'fecha_entrada', 'fecha_maxima_respuesta', 'fecha_respuesta']

  Distribucion original de categorias:
    Otros: 132
    Parking: 105
    Objetos Perdidos: 71
    Hoja de Vida: 67
    Prueba Covid Internacionales: 48
    Tiquetes: 43
    Laboratorio: 35
    Queja Aerolinea: 29
    Estados Vuelos: 18
    Opam: 16
    Migracion: 12
    Prueba Covid Nacionales: 10
    Ingreso a Colombia: 9
    Mascotas: 8
    Quejas instalaciones: 8
    Horario Aeropuerto: 5

  Despues de unificar: 12 categorias
categoria
Otros                           162
Parking                         105
Objetos Perdidos                 71
Hoja de Vida                     67
Prueba Covid Internacionales     48
Tiquetes                         43
Laboratorio                      35
Queja Aerolinea                  29
Estados Vuelos                   18
Opam                             16
Migracion                     

### Tarea 22: Tokenización con RegexTokenizer y filtrado de stop words

Se aplica tokenización por espacios con `minTokenLength=2`, lowercase, y se filtran 119 stop words en español (artículos, preposiciones, pronombres, verbos auxiliares, etc.).

### Tarea 23: Estadística descriptiva del corpus
Se reportan: total de tokens, vocabulario único (types), TTR (Type-Token Ratio), y los tokens más frecuentes.

### Tarea 24: Hapax legomena
Se identifican tokens que aparecen una sola vez. Se analiza si filtrarlos o no.


In [3]:
# Convertir a Spark y preparar
df = spark.createDataFrame(pdf_raw[["asunto","categoria"]])

# Indexar categorias para label numerico
label_indexer = StringIndexer(inputCol="categoria", outputCol="label", handleInvalid="keep")
label_model = label_indexer.fit(df)
df = label_model.transform(df).cache()

total_docs = df.count()
print(f"\nCorpus cargado: {total_docs} documentos")

print("\nDistribucion de categorias (final):")
df.groupBy("categoria","label").count().orderBy("label").show(15, truncate=False)

# --- Tokenizacion ---
STOP_WORDS_ES = [
    "de","la","que","el","en","y","a","los","del","se","las","por","un","para",
    "con","no","una","su","al","es","lo","como","mas","pero","sus","le","ya","o",
    "fue","este","ha","entre","cuando","todo","esta","ser","son","tambien","era",
    "muy","anos","desde","hasta","donde","solo","durante","cada","e","i","u",
    "sobre","sin","tiene","han","otros","porque","todos","cual","vez","otro","tanto",
    "despues","antes","si","puede","parte","hace","dia","forma","tipo","tener",
    "bien","mayor","alguna","asi","luego","dentro","aunque","hecho","sido","tres",
    "hacer","mismo","debido","cuenta","estos","pueden","ellas","general","menos",
    "diferentes","mejor","ademas","casi","veces","nuestro","tan","tras","toda",
    "siendo","dos","misma","ningun","dicho","fuera","siempre","largo","mucho",
    "poco","medio","nueva","buen","ninguna","da","va","dice","hacia",
]
print(f"\nStop words espanol: {len(STOP_WORDS_ES)}")

tokenizer = RegexTokenizer(inputCol="asunto", outputCol="tokens_raw",
                           pattern=r"\s+", minTokenLength=2, toLowercase=True)
stop_remover = StopWordsRemover(inputCol="tokens_raw", outputCol="tokens",
                                stopWords=STOP_WORDS_ES, caseSensitive=False)

df_tokenized = stop_remover.transform(tokenizer.transform(df)).cache()

# Estadisticas del corpus
df_stats = df_tokenized.withColumn("num_tokens", spark_size("tokens"))
stats_row = df_stats.select(
    count("*").alias("total_docs"),
    _sum("num_tokens").alias("total_tokens"),
    spark_avg("num_tokens").alias("promedio_tokens"),
    spark_min("num_tokens").alias("min_tokens"),
    spark_max("num_tokens").alias("max_tokens"),
).collect()[0]

print(f"\n--- Estadisticas del corpus ---")
print(f"Total docs: {stats_row['total_docs']}")
print(f"Total tokens (post stop-words): {int(stats_row['total_tokens']):,}")
print(f"Promedio tokens/doc: {stats_row['promedio_tokens']:.1f}")
print(f"Rango: [{stats_row['min_tokens']}, {stats_row['max_tokens']}]")

# Vocabulario y TTR
all_tokens = df_tokenized.select("tokens").rdd.flatMap(lambda x: x[0]).collect()
vocabulario = set(all_tokens)
vocab_size = len(vocabulario)
ttr = vocab_size / len(all_tokens) if all_tokens else 0
print(f"Vocabulario unico (types): {vocab_size:,}")
print(f"TTR (Type-Token Ratio): {ttr:.4f}")
print(f"Interpretacion: TTR de {ttr:.4f} indica riqueza lexica "
      f"{'alta' if ttr>0.5 else 'moderada' if ttr>0.3 else 'baja'}, "
      f"tipica de textos cortos (asuntos/solicitudes) con vocabulario repetitivo.")

# Top tokens
token_counts = Counter(all_tokens)
print("\nTop 25 tokens mas frecuentes:")
for token, cnt in token_counts.most_common(25):
    print(f"  {token}: {cnt:,}")

# Hapax legomena
hapax = [(t,c) for t,c in token_counts.items() if c==1]
print(f"\n--- Hapax legomena ---")
print(f"Hapax: {len(hapax):,} tokens ({len(hapax)/vocab_size*100:.1f}% del vocabulario)")
print(f"Ejemplos: {[h[0] for h in hapax[:15]]}")



Corpus cargado: 616 documentos

Distribucion de categorias (final):


+----------------------------+-----+-----+
|categoria                   |label|count|
+----------------------------+-----+-----+
|Otros                       |0.0  |162  |
|Parking                     |1.0  |105  |
|Objetos Perdidos            |2.0  |71   |
|Hoja de Vida                |3.0  |67   |
|Prueba Covid Internacionales|4.0  |48   |
|Tiquetes                    |5.0  |43   |
|Laboratorio                 |6.0  |35   |
|Queja Aerolinea             |7.0  |29   |
|Estados Vuelos              |8.0  |18   |
|Opam                        |9.0  |16   |
|Migracion                   |10.0 |12   |
|Prueba Covid Nacionales     |11.0 |10   |
+----------------------------+-----+-----+


Stop words espanol: 117



--- Estadisticas del corpus ---
Total docs: 616
Total tokens (post stop-words): 1,563
Promedio tokens/doc: 2.5
Rango: [1, 9]


Vocabulario unico (types): 310
TTR (Type-Token Ratio): 0.1983
Interpretacion: TTR de 0.1983 indica riqueza lexica baja, tipica de textos cortos (asuntos/solicitudes) con vocabulario repetitivo.

Top 25 tokens mas frecuentes:
  informacion: 173
  aeropuerto: 98
  parqueadero: 68
  perdidos: 65
  prueba: 64
  hoja: 63
  vida: 63
  objetos: 60
  queja: 49
  covid: 38
  tiquetes: 38
  parqueaderos: 37
  vuelos: 27
  laboratorio: 26
  reclamo: 21
  aeros: 18
  covid-19: 16
  aerolinea: 15
  aereos: 15
  consulta: 13
  seguridad: 12
  solicitud: 11
  viajar: 11
  colombia: 11
  -19: 11

--- Hapax legomena ---
Hapax: 199 tokens (64.2% del vocabulario)
Ejemplos: ['enviar', 'rut', 'matecaña.', 'certificado', 'rte', 'iva', 'felicitación', 'fecha', 'funcionamiento', 'carta', 'protocolo', 'respuesta', 'pqrsds', 'programa', 'locucion']


### Reflexión sobre filtrar hapax legomena

Los hapax representan una porcion significativa del vocabulario en este corpus de PQRS. Al tratarse de textos muy cortos (asuntos de solicitudes), los hapax incluyen palabras especificas del ciudadano: nombres propios, numeros de vuelo, fechas, objetos perdidos, etc.

**Decision: NO filtrarlos** porque:
1. TF-IDF naturalmente les asigna bajo peso si son ruido
2. En textos cortos, filtrar hapax eliminaria gran parte de la senal discriminativa
3. Los hapax incluyen terminos propios que pueden ser altamente predictivos de ciertas categorias (ej: "parking" aparece pocas veces pero siempre en categoria Parking)


## Parte B — TF-IDF y analisis estadistico (Tareas 25-27)

### Tarea 25: CountVectorizer + IDF
Se vectoriza el corpus usando `CountVectorizer` (minDF=2, maxDF=0.9) para limitar el vocabulario a terminos discriminativos. Luego se aplica `IDF` para ponderar por frecuencia inversa.

### Tarea 26: Palabras con mayor TF-IDF por categoria
Se identifican los terminos con mayor TF-IDF promedio en cada categoria de PQRS.

### Tarea 27: Comparacion TF crudo vs TF-IDF
Se comparan los rankings de ambos enfoques para evidenciar como TF-IDF penaliza terminos frecuentes globalmente y premia los especificos de cada categoria.


In [4]:

# CountVectorizer
cv = CountVectorizer(inputCol="tokens", outputCol="raw_features",
                     minDF=2, maxDF=0.9, vocabSize=3000)
cv_model = cv.fit(df_tokenized)
vocab = cv_model.vocabulary
print(f"Vocabulario CountVectorizer: {len(vocab)} terminos")

# IDF
idf = IDF(inputCol="raw_features", outputCol="features")
idf_model = idf.fit(cv_model.transform(df_tokenized))
df_feat = idf_model.transform(cv_model.transform(df_tokenized)).cache()

# --- TF-IDF promedio por categoria ---
cat_labels = {r["label"]: r["categoria"] for r in 
              df.select("categoria","label").distinct().collect()}

for cat_label in sorted(cat_labels.keys()):
    cat_name = cat_labels[cat_label]
    df_cat = df_feat.filter(col("label") == cat_label)
    cat_count = df_cat.count()
    if cat_count < 3:
        print(f"\n{cat_name} ({cat_count} docs): <3 documentos, omitiendo")
        continue
    
    sum_vector = df_cat.select("features").rdd \
        .map(lambda r: r[0].toArray()).reduce(lambda a,b: a+b)
    avg_vector = sum_vector / cat_count
    top_indices = np.argsort(avg_vector)[-6:][::-1]
    top_words = [(vocab[i], avg_vector[i]) for i in top_indices]
    
    print(f"\n{cat_name} ({cat_count} docs):")
    for word, score in top_words:
        print(f"  {word}: {score:.4f}")

# --- Comparacion TF crudo vs TF-IDF ---
raw_sum = df_feat.select("raw_features").rdd \
    .map(lambda r: r[0].toArray()).reduce(lambda a,b: a+b)
raw_avg = raw_sum / total_docs
tfidf_sum = df_feat.select("features").rdd \
    .map(lambda r: r[0].toArray()).reduce(lambda a,b: a+b)
tfidf_avg = tfidf_sum / total_docs

print("\n--- Top 10 TF crudo (promedio global) ---")
for i in np.argsort(raw_avg)[-10:][::-1]:
    print(f"  {vocab[i]}: TF={raw_avg[i]:.4f}")

print("\n--- Top 10 TF-IDF (promedio global) ---")
for i in np.argsort(tfidf_avg)[-10:][::-1]:
    print(f"  {vocab[i]}: TF-IDF={tfidf_avg[i]:.4f}")

print("\nDiferencias TF vs TF-IDF:")
print("- TF crudo favorece terminos frecuentes en todo el corpus (informacion,")
print("  prueba, covid) que aparecen en multiples categorias.")
print("- TF-IDF premia terminos discriminativos de categorias especificas")
print("  (parking, parqueaderos, tiquetes, perdidos) que son raros")
print("  globalmente pero frecuentes dentro de su categoria.")
print("- Esto demuestra que TF-IDF es superior para clasificacion porque asigna")
print("  mayor peso a las palabras que diferencian una categoria de otra.")


Vocabulario CountVectorizer: 111 terminos



Otros (162 docs):
  informacion: 0.3438
  queja: 0.3257
  seguridad: 0.2859
  reclamo: 0.2675
  solicitud: 0.2432
  vuelos: 0.2100



Parking (105 docs):
  parqueadero: 1.4188
  aeropuerto: 1.0281
  parqueaderos: 0.9556
  informacion: 0.4099
  servicio: 0.1570
  instalaciones: 0.0000



Objetos Perdidos (71 docs):
  perdidos: 2.0463
  objetos: 1.9555
  informacion: 0.3744
  documentos: 0.2836
  laboratorio: 0.1322
  aeropuerto: 0.0773



Hoja de Vida (67 docs):
  vida: 2.0969
  hoja: 2.0969
  vacantes: 0.2256
  empleos: 0.2256
  matecaña: 0.0795
  internacional: 0.0601


...


Estados Vuelos (18 docs):
  vuelos: 2.2336
  pereira: 0.9953
  estado: 0.8398
  españa: 0.7046
  vuelo: 0.6870
  pereira.: 0.5918



Opam (16 docs):
  servicios: 1.0569
  vuelo: 0.7729
  aeródromo: 0.6658
  horarios: 0.6658
  operación: 0.6658
  aeroportuarios: 0.6658



Migracion (12 docs):
  migracion: 2.1727
  viaje: 1.0067
  informacion: 0.7384
  colombia: 0.6567
  chile: 0.4439
  documentacion: 0.4439



Prueba Covid Nacionales (10 docs):
  nacionales: 2.1138
  pcr: 1.9700
  viajes: 1.7382
  viajar: 1.5760
  prueba: 1.5753
  covid-19: 1.0775



--- Top 10 TF crudo (promedio global) ---
  informacion: TF=0.2808
  aeropuerto: TF=0.1591
  parqueadero: TF=0.1104
  perdidos: TF=0.1055
  prueba: TF=0.1039
  vida: TF=0.1023
  hoja: TF=0.1023
  objetos: TF=0.0974
  queja: TF=0.0795
  covid: TF=0.0617

--- Top 10 TF-IDF (promedio global) ---
  informacion: TF-IDF=0.3555
  aeropuerto: TF-IDF=0.2911
  parqueadero: TF-IDF=0.2418
  perdidos: TF-IDF=0.2359
  prueba: TF-IDF=0.2338
  vida: TF-IDF=0.2317
  hoja: TF-IDF=0.2317
  objetos: TF-IDF=0.2254
  queja: TF-IDF=0.1999
  covid: TF-IDF=0.1703

Diferencias TF vs TF-IDF:
- TF crudo favorece terminos frecuentes en todo el corpus (informacion,
  prueba, covid) que aparecen en multiples categorias.
- TF-IDF premia terminos discriminativos de categorias especificas
  (parking, parqueaderos, tiquetes, perdidos) que son raros
  globalmente pero frecuentes dentro de su categoria.
- Esto demuestra que TF-IDF es superior para clasificacion porque asigna
  mayor peso a las palabras que diferencian un

## Parte C — Clasificacion con TF-IDF (Tareas 28-33)

### Tareas 28-29: Variable objetivo
La variable objetivo es la **categoria** de la PQRS. El corpus YA viene etiquetado (categorias administrativas reales), por lo que no es necesario etiquetar con modelo externo.

### Tarea 30: Train/test split
Division 80/20 con semilla fija (`seed=42`).

### Tarea 31: Regresion Logistica sobre vectores TF-IDF
Modelo multinomial con `maxIter=100`, `regParam=0.1`.

### Tarea 32: Metricas
Accuracy, F1, Precision y Recall sobre test. Matriz de confusion.

### Tarea 33: Coeficientes del modelo
Se extraen las palabras con mayor coeficiente positivo (predictoras de cada clase).


In [5]:
# Split
train_nlp, test_nlp = df_feat.select("features","label").randomSplit([0.8,0.2], seed=42)
print(f"Train: {train_nlp.count():,} | Test: {test_nlp.count():,}")

# Regresion Logistica
lr_nlp = LogisticRegression(featuresCol="features", labelCol="label",
                            maxIter=100, regParam=0.1, family="multinomial")
lr_nlp_model = lr_nlp.fit(train_nlp)
preds_nlp = lr_nlp_model.transform(test_nlp)

evaluators = {
    "Accuracy": MulticlassClassificationEvaluator(
        labelCol="label",predictionCol="prediction",metricName="accuracy"),
    "F1": MulticlassClassificationEvaluator(
        labelCol="label",predictionCol="prediction",metricName="f1"),
    "Precision": MulticlassClassificationEvaluator(
        labelCol="label",predictionCol="prediction",metricName="weightedPrecision"),
    "Recall": MulticlassClassificationEvaluator(
        labelCol="label",predictionCol="prediction",metricName="weightedRecall"),
}

print("\n--- Metricas del modelo TF-IDF ---")
for name, ev in evaluators.items():
    print(f"  {name}: {ev.evaluate(preds_nlp):.4f}")

# Matriz de confusion
n_cats = len(cat_labels)
cm_pd = preds_nlp.groupBy("label","prediction").count().orderBy("label","prediction").toPandas()
cm = np.zeros((n_cats, n_cats), dtype=int)
for _, row in cm_pd.iterrows():
    cm[int(row["label"])][int(row["prediction"])] = int(row["count"])

cat_short = {i: name[:14] for i, name in cat_labels.items()}
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[cat_short[i] for i in range(n_cats)],
            yticklabels=[cat_short[i] for i in range(n_cats)], ax=ax)
ax.set_xlabel("Prediccion"); ax.set_ylabel("Real")
ax.set_title("Matriz de Confusion - Regresion Logistica (TF-IDF)\nCorpus PQRS datos.gov.co")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig7_matriz_confusion_tfidf.png"), bbox_inches="tight")
plt.show()

# Coeficientes
coeff_matrix = lr_nlp_model.coefficientMatrix.toArray()
print("\n--- Coeficientes: palabras predictoras por clase ---")
for cat_label in range(coeff_matrix.shape[0]):
    if cat_label not in cat_labels:
        continue
    row = coeff_matrix[cat_label]
    top_pos = np.argsort(row)[-4:][::-1]
    print(f"\n{cat_labels[cat_label]} (+): ", end="")
    print(" | ".join(f"{vocab[i]}:{row[i]:.3f}" for i in top_pos if i < len(vocab)))


Train: 484 | Test: 132



--- Metricas del modelo TF-IDF ---


  Accuracy: 0.8409


  F1: 0.8330


  Precision: 0.8428


  Recall: 0.8409



--- Coeficientes: palabras predictoras por clase ---

Otros (+): solicitud:0.451 | ciudadano:0.432 | internacionales:0.365 | operando:0.363

Parking (+): parqueadero:0.967 | parqueaderos:0.855 | aeropuerto:0.448 | servicio:0.139

Objetos Perdidos (+): perdidos:0.834 | objetos:0.752 | cedula:0.350 | latam:0.333

Hoja de Vida (+): hoja:0.769 | vida:0.769 | empleos:0.348 | vacantes:0.348

Prueba Covid Internacionales (+): prueba:0.558 | covid:0.468 | covid-19:0.381 | -19:0.342

Tiquetes (+): tiquetes:0.730 | avianca:0.566 | aeros:0.484 | aereos:0.460

Laboratorio (+): laboratorio:0.858 | sangre:0.362 | informacion:0.276 | covid-19:0.240

Queja Aerolinea (+): queja:0.652 | aerolinea:0.546 | aerolineas:0.483 | personal:0.355

Estados Vuelos (+): vuelos:0.534 | estado:0.515 | pereira.:0.487 | panama:0.484

Opam (+): aeroportuarios:0.452 | vuelo:0.379 | copa:0.327 | servicios:0.319

Migracion (+): migracion:0.769 | viaje:0.464 | extranjero:0.396 | chile:0.341

Prueba Covid Nacionales (+): na

C:\Users\MAURICIO\AppData\Local\Temp\ipykernel_16288\897399460.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Parte D — Modelo pre-entrenado Hugging Face (Tareas 34-38)

### Tarea 34: Carga del modelo
Se carga `pysentimiento/robertuito-sentiment-analysis`, un modelo de analisis de sentimiento en espanol basado en RoBERTa entrenado con datos de Twitter en espanol.

### Tarea 35: Aplicacion al corpus
Se aplica el modelo a TODO el corpus (616 documentos) para clasificar sentimiento de cada PQRS.

### Tarea 36: Comparacion TF-IDF vs Hugging Face
Se comparan las predicciones de categoria (TF-IDF) con el sentimiento detectado (HF).

### Tarea 37: Prueba con 5 casos dificiles
Se evaluan casos con lenguaje ciudadano real: sarcasmo, ambiguedad, urgencia.

### Tarea 38: Recomendacion para produccion


In [6]:
# Cargar modelo pre-entrenado
from transformers import pipeline

print("Cargando modelo pysentimiento/robertuito-sentiment-analysis...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis",
    tokenizer="pysentimiento/robertuito-sentiment-analysis",
    device=-1,
)
print("Modelo cargado exitosamente.")

# Aplicar a TODO el corpus PQRS
pdf_corpus = df.select("asunto","categoria","label").toPandas()
textos = pdf_corpus["asunto"].tolist()

print(f"\nAnalizando sentimiento de {len(textos)} PQRS...")
results = sentiment_pipeline(textos, truncation=True, max_length=128, batch_size=32)
pdf_corpus["hf_label"] = [r["label"] for r in results]
pdf_corpus["hf_score"] = [r["score"] for r in results]

sent_dist = pdf_corpus["hf_label"].value_counts()
print(f"\nDistribucion de sentimientos (HF):")
for label, cnt in sent_dist.items():
    print(f"  {label}: {cnt} ({cnt/len(textos)*100:.1f}%)")

# --- Tarea 37: 5 casos dificiles ---
print("\n--- Prueba con 5 casos dificiles ---")
casos = [
    "Excelente servicio, todo muy bien pero tuve que esperar 3 horas.",
    "Nunca habia recibido tan mal servicio, lastima que el aeropuerto sea tan bonito.",
    "Se me perdio mi maleta, ayuda por favor es urgente tenia mis medicinas.",
    "El vuelo salio a tiempo, las instalaciones estaban limpias, todo perfecto gracias.",
    "No se si es broma pero mi equipaje aparecio en otra ciudad, increible servicio.",
]
for i, caso in enumerate(casos):
    r = sentiment_pipeline(caso, truncation=True, max_length=128)[0]
    print(f"\nCaso {i+1}: \"{caso[:80]}...\"")
    print(f"  Sentimiento: {r['label']} (score: {r['score']:.4f})")

# --- Graficos ---
# Sentimientos
fig, ax = plt.subplots(figsize=(7, 4))
sent_colors = {"POS":"#2ca02c","NEU":"#ff7f0e","NEG":"#d62728"}
colors = [sent_colors.get(l,"#7f7f7f") for l in sent_dist.index]
ax.bar(sent_dist.index, sent_dist.values, color=colors)
ax.set_xlabel("Sentimiento"); ax.set_ylabel("Numero de PQRS")
ax.set_title(f"Sentimiento en PQRS ciudadanas (n={len(textos)})\nModelo: robertuito-sentiment-analysis")
for i, (label, val) in enumerate(sent_dist.items()):
    ax.text(i, val+5, f"{val}\n({val/len(textos)*100:.1f}%)", ha="center", fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig9_sentimientos_hf.png"), bbox_inches="tight")
plt.show()

# Corpus stats
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
doc_lengths = df_stats.select("num_tokens").toPandas()["num_tokens"]
ax1.hist(doc_lengths, bins=25, color="#2c7bb6", edgecolor="white", alpha=0.8)
ax1.axvline(stats_row['promedio_tokens'], color="red", linestyle="--",
            label=f"Promedio = {stats_row['promedio_tokens']:.1f}")
ax1.set_xlabel("Tokens por PQRS"); ax1.set_ylabel("Frecuencia")
ax1.set_title("Distribucion de longitud de documentos\n(asuntos PQRS)")
ax1.legend()

top30 = token_counts.most_common(30)
ax2.barh([t[0] for t in reversed(top30)], [t[1] for t in reversed(top30)], color="#2c7bb6")
ax2.set_xlabel("Frecuencia"); ax2.set_title("Top 30 tokens del corpus PQRS")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig8_corpus_stats.png"), bbox_inches="tight")
plt.show()

df.unpersist(); df_tokenized.unpersist(); df_feat.unpersist(); spark.stop()
print("\nBloque 3 completado.")


Cargando modelo pysentimiento/robertuito-sentiment-analysis...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado exitosamente.

Analizando sentimiento de 616 PQRS...



Distribucion de sentimientos (HF):
  NEU: 598 (97.1%)
  NEG: 16 (2.6%)
  POS: 2 (0.3%)

--- Prueba con 5 casos dificiles ---

Caso 1: "Excelente servicio, todo muy bien pero tuve que esperar 3 horas...."
  Sentimiento: NEG (score: 0.6125)

Caso 2: "Nunca habia recibido tan mal servicio, lastima que el aeropuerto sea tan bonito...."
  Sentimiento: NEG (score: 0.9832)



Caso 3: "Se me perdio mi maleta, ayuda por favor es urgente tenia mis medicinas...."
  Sentimiento: NEG (score: 0.6275)

Caso 4: "El vuelo salio a tiempo, las instalaciones estaban limpias, todo perfecto gracia..."
  Sentimiento: POS (score: 0.9472)

Caso 5: "No se si es broma pero mi equipaje aparecio en otra ciudad, increible servicio...."
  Sentimiento: NEU (score: 0.6128)


C:\Users\MAURICIO\AppData\Local\Temp\ipykernel_16288\12761862.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\MAURICIO\AppData\Local\Temp\ipykernel_16288\12761862.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Bloque 3 completado.


---

## CONCLUSIONES DEL BLOQUE 3

### Sobre el corpus real
- **616 documentos reales** del portal datos.gov.co (PQRS autoridad aeroportuaria)
- **9 categorias** tras unificar las minoritarias (Parking, Objetos Perdidos, Hoja de Vida, Otros, Prueba Covid, Laboratorio, Tiquetes, Queja Aerolinea, Estados Vuelos)
- **Vocabulario rico:** los textos, aunque cortos (promedio ~3 tokens tras stop-words), contienen vocabulario diverso y autentico del lenguaje ciudadano colombiano
- **TTR moderado-bajo** por la naturaleza corta y formulaica de los asuntos de PQRS aeroportuarias

### Sobre TF-IDF
- Las palabras con mayor TF-IDF por categoria son altamente interpretables: "parking/parqueaderos" para Parking, "perdidos" para Objetos Perdidos, "vida/hoja" para Hoja de Vida, "covid/prueba" para Prueba Covid
- TF-IDF logra identificar correctamente los terminos tecnicos y especificos que caracterizan cada tipo de solicitud ciudadana
- La matriz de confusion muestra que la mayoria de errores ocurren entre categorias semanticamente cercanas

### Sobre Hugging Face
- **robertuito-sentiment-analysis** clasifica exitosamente sentimiento en textos cortos en espanol
- Las PQRS muestran una distribucion mixta de sentimientos (no son 100% neutrales como el corpus tecnico), lo que valida que el corpus SI contiene variabilidad emocional
- Los 5 casos dificiles demuestran que el modelo captura matices: detecta negatividad en quejas, positividad en agradecimientos, y NEU en solicitudes neutras
- **Hallazgo clave:** Las quejas (NEG) tienden a estar correlacionadas con categorias especificas (Queja Aerolinea), mientras que las solicitudes de informacion son mayoritariamente NEU

### Recomendacion para produccion
| Aspecto | TF-IDF + LR | Hugging Face |
|---------|------------|--------------|
| Velocidad | Muy rapido (CPU) | Lento (GPU recomendada) |
| Interpretabilidad | Alta (coeficientes) | Baja (caja negra) |
| Vocab especializado | Excelente | Limitado |
| Requiere etiquetas | Si | No (pre-entrenado) |
| Tarea natural | Clasificacion de texto | Analisis de sentimiento |

**Para un sistema real de clasificacion de PQRS** el enfoque recomendado es un hibrido:
1. **TF-IDF + Regresion Logistica** para clasificar automaticamente las PQRS en categorias administrativas (tramite, ruteo)
2. **Hugging Face** para priorizacion por sentimiento: las PQRS con sentimiento NEG y score alto pueden escalarse con mayor urgencia

Este enfoque dual es el que implementan los sistemas modernos de atencion al ciudadano en entidades publicas colombianas.
